[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/13_gpt2_block.ipynb)

# 🔴 Hard: GPT-2 Transformer Block

Implement a full **GPT-2 style Transformer block** — combining everything you've learned.

### Architecture (Pre-Norm)
```
x = x + causal_self_attention(ln1(x))
x = x + mlp(ln2(x))
```

### Signature
```python
class GPT2Block(nn.Module):
    def __init__(self, d_model: int, num_heads: int): ...
    def forward(self, x: torch.Tensor) -> torch.Tensor: ...
```

### Requirements
- Inherit from `nn.Module`
- `self.ln1`, `self.ln2`: `nn.LayerNorm(d_model)`
- `self.W_q`, `self.W_k`, `self.W_v`, `self.W_o`: `nn.Linear` for attention
- `self.mlp`: `nn.Sequential(Linear(d, 4d), GELU(), Linear(4d, d))`
- Attention must be **causal** (mask future positions)
- Pre-norm architecture (LayerNorm *before* attention and MLP)
- Residual connections around both attention and MLP

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [1]:
import torch
import torch.nn as nn
import math

/usr/local/lib/python3.11/site-packages/torch/_subclasses/functional_tensor.py:307: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE
class GPT2Block(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        self.mlp = nn.Sequential(
            nn.Linear(self.d_model, 4*self.d_model),
            nn.GELU(),
            nn.Linear(4 * self.d_model, self.d_model)
        )

    def _atten(self, x):
        batch_size, seq_len, _ = x.shape
        d_head = self.d_model // self.num_heads
        Q = self.W_q(x).view(batch_size, seq_len, self.num_heads, d_head).transpose(1, 2)
        K = self.W_k(x).view(batch_size, seq_len, self.num_heads, d_head).transpose(1, 2)
        V = self.W_v(x).view(batch_size, seq_len, self.num_heads, d_head).transpose(1, 2)
        atten_score = Q @ K.transpose(-1, -2)
        mask = torch.tril(torch.ones(seq_len, seq_len))
        atten_score = atten_score.masked_fill(mask == 0, float('-inf'))
        atten_score = torch.softmax(atten_score / math.sqrt(d_head), dim = -1)
        context_vect = atten_score @ V
        context_vect = context_vect.transpose(1,2)
        context_vect = context_vect.contiguous().view(batch_size, seq_len, self.d_model)
        return self.W_o(context_vect)

    def forward(self, x):
        # Pre-norm + causal attention + MLP with residuals
        x = x + self._atten(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

In [21]:
transformer_block = GPT2Block(d_model = 128, num_heads = 4)

In [22]:
transformer_block.parameters

<bound method Module.parameters of GPT2Block(
  (ln1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
  (ln2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
  (W_q): Linear(in_features=128, out_features=128, bias=True)
  (W_k): Linear(in_features=128, out_features=128, bias=True)
  (W_v): Linear(in_features=128, out_features=128, bias=True)
  (W_o): Linear(in_features=128, out_features=128, bias=True)
  (mlp): Sequential(
    (0): Linear(in_features=128, out_features=512, bias=True)
    (1): GELU(approximate='none')
    (2): Linear(in_features=512, out_features=128, bias=True)
  )
)>

In [23]:
# 🧪 Debug
torch.manual_seed(0)
block = GPT2Block(d_model=64, num_heads=4)
x = torch.randn(2, 8, 64)
out = block(x)
print("Output shape:", out.shape)           # (2, 8, 64)
print("Is nn.Module?", isinstance(block, nn.Module))
print("Params:", sum(p.numel() for p in block.parameters()))

Output shape: torch.Size([2, 8, 64])
Is nn.Module? True
Params: 49984


In [24]:
from torch_judge import check
check('gpt2_block')


🧪 Testing: GPT-2 Transformer Block (Hard)
──────────────────────────────────────────────────
  ✅ [1/5] Output shape (4.4ms)
  ✅ [2/5] Has LayerNorm (pre-norm architecture) (0.9ms)
  ✅ [3/5] MLP has 4x expansion with GELU (0.9ms)
  ✅ [4/5] Causal masking — future doesn't affect past (4.4ms)
  ✅ [5/5] Gradient flow to all parameters (5.4ms)
──────────────────────────────────────────────────
  🎉 All 5 tests passed! (16.0ms total)
  Progress saved. Run status() to see your dashboard.

